In [760]:
import requests
import json
import numpy as np
import dateutil as du
import datetime
from datetime import tzinfo
#from metpy.units import units

In [761]:
stationID = input("Enter in the station ID:").upper()
requestedMonth = int(input("Enter in the month you want (int):"))
requestedDay = int(input("Enter in the date you want (int):"))

In [762]:
# If the station is reporting null for the temperature, replace it with np.nan.
# This allows the use of math and of max/mins.
def isStationNull(val):
    if(val == None):
        return np.nan
    else:
        return val

# If the station is reporting null for the rainfall, replace it with 0. 
def isStationNullPrecip(val):
    if(val == None):
        return 0
    else:
        return val


# This retrieves all of the station data from a specified station.
def stationRetrieval(station):
    api_urlStationData = "https://api.weather.gov/stations/" + station + "/observations"
    responseData = requests.get(api_urlStationData)

    dataStation = json.loads(json.dumps(responseData.json()))

    return dataStation

# This grabs the temperature and puts it into a list. Checks for null values.
def temperatureFinder(dataInput):
    dataTemperature = []

    for i in dataInput:
        initTemp = i["properties"]["temperature"]["value"]

        dataTemperature.append(isStationNull(initTemp))

    return dataTemperature

# This grabs the precipitation and enters it into a list. Also checks for null values.
def precipitationFinder(dataInput):
    dataPrecipitation = []

    for i in dataInput:
        initPrecip = i["properties"]["precipitationLastHour"]["value"]

        dataPrecipitation.append(isStationNullPrecip(initPrecip))

    return dataPrecipitation

# This grabs the wind speed and enters into a list. Checks yet again for null values.
def windSpeedFinder(dataInput):
    dataWindSpeed = []

    for i in dataInput:
        initWinds = i["properties"]["windSpeed"]["value"]

        dataWindSpeed.append(isStationNull(initWinds))

    return dataWindSpeed

# Needed to find and sort the dates. This finds the dates that are within the bounds set by the user.
def dateFinder(dataInput):
    dataDates = []
    dataReturned = []
    count1 = 0
    indexCount = -1

    # This narrows down the range of dates to specifically the ones requested
    for i in dataInput:
        initDate = i["properties"]["timestamp"]
        dateProcessed = du.parser.parse(initDate)

        # Checks if the current date is between 06Z of the day requested and 06Z of the next day.
        if(dateProcessed < datetime.datetime(2023, requestedMonth, requestedDay + 1, 6, 00, tzinfo=datetime.timezone.utc) and \
           dateProcessed > datetime.datetime(2023, requestedMonth, requestedDay, 6, 00, tzinfo=datetime.timezone.utc)):
            dataDates.append(dateProcessed)

            # This sets where the list starts at for trimming down the later parts of the lists.
            if(indexCount == -1):
                indexCount = count1

        count1 += 1

        # Since the program needs both the dates and index of the first value, return it as a list.
        dataReturned = [dataDates, int(indexCount)]

    return dataReturned

def celsiusToFahrenheit(tempC):
    tempF = tempC * 1.8 + 32
    return tempF

def kmhToKnots(windKMH):
    windKts = windKMH / 1.852
    return windKts

In [763]:
# This runs all of the functions that get and process the data.
dataStation = stationRetrieval(stationID)
dataTemp = temperatureFinder(dataStation["features"])
dataPrecip = precipitationFinder(dataStation["features"])
dataWind = windSpeedFinder(dataStation["features"])
dataDate = dateFinder(dataStation["features"])

# Trims down the lists to just the day that is requested.
dataTempTrim = dataTemp[int(dataDate[1]) : dataDate[1]+len(dataDate[0])]
dataPrecipTrim = dataPrecip[int(dataDate[1]) : dataDate[1]+len(dataDate[0])]
dataWindTrim = dataWind[int(dataDate[1]) : dataDate[1]+len(dataDate[0])]

In [764]:
print("High: " + str(celsiusToFahrenheit(max(dataTempTrim)))[:5] + "°F")
print("Low: " + str(celsiusToFahrenheit(min(dataTempTrim)))[:5] + "°F")

# Annoyingly, the NWS reports the rainfall in whole centimeters. Yeah, I know.
# This results in significant overestimates in rainfall.
# A workaround I found was to divide by 50 rather than 25.4 to convert between millimeters and inches.
# Don't put much weight into this approximation, to be honest.
print("Rainfall: " + str(sum(dataPrecipTrim)/50) + " inches")

# Winds are also unreliable. Might try to find a more reliable and consistent API...
print("Wind speed: " + str(kmhToKnots(max(dataWindTrim)))[:5] + " knots")

High: 84.92°F
Low: 64.94°F
Rainfall: 0.0 inches
Wind speed: 13.02 knots
